<a href="https://colab.research.google.com/github/TanVi3001/Leakage_Data/blob/main/Flower_leakageData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Bài toán Flower Classification

### Dataset
- Flower Classification
- 14 classes

### Bài toán:
              Image

                ↓

                CNN

                ↓
                
              Flower class

## Load Flower dataset

### Giả sử có ảnh:

rose_001.jpg

**Ta augment:**

- rose_001.jpg
- rose_001_flip.jpg
- rose_001_rotate.jpg
- rose_001_crop.jpg

**Sau đó mới random split:**

- rose_001.jpg --> TRAIN


- rose_001_flip.jpg --> TEST


**Nhìn bề ngoài:**

- 2 filename khác nhau

**Nhưng thực tế:**

- cùng một ảnh gốc
- cùng hoa
- cùng background
- cùng ánh sáng
- cùng texture

--> **Model gần như đã nhìn thấy test sample.**

--> **Đây là Augmentation Leakage / Duplicate Leakage.**

## Load Flower dataset

In [1]:
!pip install -q kaggle

In [2]:
!kaggle datasets download \
    -d marquis03/flower-classification \
    -p /content/flowers \
    --unzip

Dataset URL: https://www.kaggle.com/datasets/marquis03/flower-classification
License(s): apache-2.0
100% 205M/205M [00:02<00:00, 104MB/s]



In [3]:
import os

print(os.listdir("/content/flowers"))

['classname.txt', 'train.csv', 'train', 'val', 'val.csv']


In [4]:
for root, dirs, files in os.walk("/content/flowers"):
    level = root.replace("/content/flowers", "").count(os.sep)

    if level <= 2:
        print(root)

/content/flowers
/content/flowers/train
/content/flowers/train/bellflower
/content/flowers/train/water_lily
/content/flowers/train/black_eyed_susan
/content/flowers/train/california_poppy
/content/flowers/train/iris
/content/flowers/train/carnation
/content/flowers/train/tulip
/content/flowers/train/sunflower
/content/flowers/train/astilbe
/content/flowers/train/calendula
/content/flowers/train/rose
/content/flowers/train/common_daisy
/content/flowers/train/coreopsis
/content/flowers/train/dandelion
/content/flowers/val
/content/flowers/val/bellflower
/content/flowers/val/water_lily
/content/flowers/val/black_eyed_susan
/content/flowers/val/california_poppy
/content/flowers/val/iris
/content/flowers/val/carnation
/content/flowers/val/tulip
/content/flowers/val/sunflower
/content/flowers/val/astilbe
/content/flowers/val/calendula
/content/flowers/val/rose
/content/flowers/val/common_daisy
/content/flowers/val/coreopsis
/content/flowers/val/dandelion


In [5]:
flower_root = "/content/flowers/train"

In [6]:
from torchvision import datasets

base_dataset = datasets.ImageFolder(
    flower_root
)

print("Số ảnh:", len(base_dataset))
print("Các lớp:", base_dataset.classes)
print("Số lớp:", len(base_dataset.classes))

Số ảnh: 13642
Các lớp: ['astilbe', 'bellflower', 'black_eyed_susan', 'calendula', 'california_poppy', 'carnation', 'common_daisy', 'coreopsis', 'dandelion', 'iris', 'rose', 'sunflower', 'tulip', 'water_lily']
Số lớp: 14


## Split dữ liệu gốc trước

### Ta chia:

- 70% Train
- 15% Validation
- 15% Test

In [7]:
from collections import defaultdict
import torch

In [8]:
def stratified_split(
    targets,
    train_ratio=0.7,
    val_ratio=0.15,
    seed=42
):

    class_indices = defaultdict(list)

    for idx, label in enumerate(targets):
        class_indices[label].append(idx)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_idx = []
    val_idx = []
    test_idx = []

    for label, indices in class_indices.items():

        order = torch.randperm(
            len(indices),
            generator=generator
        ).tolist()

        indices = [
            indices[i]
            for i in order
        ]

        n = len(indices)

        n_train = int(
            n * train_ratio
        )

        n_val = int(
            n * val_ratio
        )

        train_idx.extend(
            indices[:n_train]
        )

        val_idx.extend(
            indices[
                n_train:n_train+n_val
            ]
        )

        test_idx.extend(
            indices[
                n_train+n_val:
            ]
        )

    return (
        train_idx,
        val_idx,
        test_idx
    )

In [9]:
train_idx, val_idx, test_idx = (
    stratified_split(
        base_dataset.targets
    )
)

## Kiểm tra leakage

In [10]:
assert set(train_idx).isdisjoint(
    val_idx
)

assert set(train_idx).isdisjoint(
    test_idx
)

assert set(val_idx).isdisjoint(
    test_idx
)

print(
    "No index overlap"
)

No index overlap


## Transform Train

In [11]:
from torchvision import transforms

In [12]:
train_transform = transforms.Compose([

    transforms.RandomResizedCrop(
        224
    ),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(
        15
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

## Transform Validation/Test

In [13]:
eval_transform = transforms.Compose([

    transforms.Resize(
        256
    ),

    transforms.CenterCrop(
        224
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

## Wrapper Dataset

In [14]:
from torch.utils.data import Dataset

In [15]:
class TransformSubset(Dataset):

    def __init__(
        self,
        dataset,
        indices,
        transform
    ):

        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):

        return len(
            self.indices
        )

    def __getitem__(self, idx):

        real_idx = self.indices[idx]

        image, label = (
            self.dataset[
                real_idx
            ]
        )

        image = self.transform(
            image
        )

        return image, label

## Tạo dataset:

In [16]:
train_flower = TransformSubset(
    base_dataset,
    train_idx,
    train_transform
)

val_flower = TransformSubset(
    base_dataset,
    val_idx,
    eval_transform
)

test_flower = TransformSubset(
    base_dataset,
    test_idx,
    eval_transform
)

## Dataloader

In [17]:
from torch.utils.data import DataLoader

In [18]:
train_loader = DataLoader(
    train_flower,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_flower,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_flower,
    batch_size=64,
    shuffle=False
)

## CNN

In [19]:
import torch
import torch.nn as nn

class FlowerCNN(nn.Module):

    def __init__(self, num_classes=14):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # Block 2
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # Block 3
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # Block 4
            nn.Conv2d(
                in_channels=128,
                out_channels=256,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        self.pool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.pool(x)

        x = self.classifier(x)

        return x

## Tạo model CNN

In [20]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Đang sử dụng:", device)

Đang sử dụng: cuda


In [21]:
num_classes = len(
    base_dataset.classes
)

print(
    "Số lớp:",
    num_classes
)

model = FlowerCNN(
    num_classes=num_classes
)

model = model.to(device)

print(model)

Số lớp: 14
FlowerCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_

## Transform cũng nên đổi một chút

### Train transform

In [22]:
from torchvision import transforms

train_transform = transforms.Compose([

    transforms.Resize(
        (128, 128)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        15
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

## Validation/Test:

In [23]:
eval_transform = transforms.Compose([

    transforms.Resize(
        (128, 128)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

## Kiểm tra 1 batch trước khi train

In [24]:
images, labels = next(
    iter(train_loader)
)

print(
    "Image shape:",
    images.shape
)

print(
    "Label shape:",
    labels.shape
)

Image shape: torch.Size([32, 3, 224, 224])
Label shape: torch.Size([32])


## Kiểm tra CNN

In [25]:
images = images.to(device)

outputs = model(images)

print(
    outputs.shape
)

torch.Size([32, 14])


## Loss Function

In [26]:
criterion = nn.CrossEntropyLoss()

## Optimizer

In [27]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

## Train CNN Flower CLEAN

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

# =========================================================
# 1. DEVICE
# =========================================================
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# =========================================================
# 2. ĐƯA MODEL LÊN DEVICE
# =========================================================
model = model.to(device)


# =========================================================
# 3. LOSS + OPTIMIZER
# =========================================================
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)


# =========================================================
# 4. HÀM TRAIN 1 EPOCH
# =========================================================
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        # Xóa gradient
        optimizer.zero_grad()

        # Forward
        outputs = model(images)

        # Loss
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()

        # Update weights
        optimizer.step()

        # Cộng loss
        running_loss += (
            loss.item() * images.size(0)
        )

        # Prediction
        preds = outputs.argmax(dim=1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


# =========================================================
# 5. HÀM EVALUATE
# =========================================================
@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion
):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        running_loss += (
            loss.item() * images.size(0)
        )

        preds = outputs.argmax(dim=1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


# =========================================================
# 6. KIỂM TRA TRƯỚC KHI TRAIN
# =========================================================
print("train_one_epoch tồn tại:", callable(train_one_epoch))
print("evaluate tồn tại:", callable(evaluate))

print("Số batch train:", len(train_loader))
print("Số batch validation:", len(val_loader))


# =========================================================
# 7. TRAIN CNN FLOWER CLEAN
# =========================================================
num_epochs = 5

best_val_acc = 0.0
best_model_state = None

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion
    )

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        best_model_state = copy.deepcopy(
            model.state_dict()
        )


# =========================================================
# 8. LOAD MODEL TỐT NHẤT
# =========================================================
model.load_state_dict(
    best_model_state
)

print("\nTraining hoàn tất!")
print(
    f"Best Validation Accuracy: "
    f"{best_val_acc:.4f}"
)

Device: cuda
train_one_epoch tồn tại: True
evaluate tồn tại: True
Số batch train: 299
Số batch validation: 32
Epoch 01/5 | Train Loss: 1.9954 | Train Acc: 0.2936 | Val Loss: 1.6690 | Val Acc: 0.3974
Epoch 02/5 | Train Loss: 1.7520 | Train Acc: 0.3789 | Val Loss: 1.6765 | Val Acc: 0.4087
Epoch 03/5 | Train Loss: 1.6282 | Train Acc: 0.4270 | Val Loss: 1.4105 | Val Acc: 0.4760
Epoch 04/5 | Train Loss: 1.5459 | Train Acc: 0.4558 | Val Loss: 1.3630 | Val Acc: 0.5093
Epoch 05/5 | Train Loss: 1.5123 | Train Acc: 0.4665 | Val Loss: 1.2437 | Val Acc: 0.5368

Training hoàn tất!
Best Validation Accuracy: 0.5368


## Test Flower CLEAN

In [29]:
# =========================================================
# TEST FLOWER CLEAN
# =========================================================

test_loss_clean, test_acc_clean = evaluate(
    model,
    test_loader,
    criterion
)

print("=" * 50)
print("FLOWER CLEAN - FINAL TEST RESULT")
print("=" * 50)

print(
    f"Test Loss     : {test_loss_clean:.4f}"
)

print(
    f"Test Accuracy : {test_acc_clean:.4f}"
)

print(
    f"Test Accuracy : {test_acc_clean * 100:.2f}%"
)

FLOWER CLEAN - FINAL TEST RESULT
Test Loss     : 1.2279
Test Accuracy : 0.5407
Test Accuracy : 54.07%


In [30]:
flower_clean_result = {
    "best_val_acc": best_val_acc,
    "test_loss": test_loss_clean,
    "test_acc": test_acc_clean
}

print(flower_clean_result)

{'best_val_acc': 0.5368007850834151, 'test_loss': 1.2278659655328634, 'test_acc': 0.5407371483996121}


## Kiểm tra accuracy theo từng class

In [31]:
import torch

def get_predictions(
    model,
    loader
):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds = outputs.argmax(
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    return all_labels, all_preds

In [32]:
clean_labels, clean_preds = get_predictions(
    model,
    test_loader
)

In [33]:
from sklearn.metrics import classification_report

print(
    classification_report(
        clean_labels,
        clean_preds,
        target_names=base_dataset.classes,
        digits=4
    )
)

                  precision    recall  f1-score   support

         astilbe     0.5026    0.8636    0.6355       110
      bellflower     0.6316    0.2727    0.3810       132
black_eyed_susan     0.4006    0.9329    0.5605       149
       calendula     0.4241    0.4379    0.4309       153
california_poppy     0.8519    0.1494    0.2541       154
       carnation     0.5323    0.2357    0.3267       140
    common_daisy     0.5746    0.7027    0.6322       148
       coreopsis     0.3086    0.1603    0.2110       156
       dandelion     0.8028    0.7261    0.7625       157
            iris     0.6168    0.6561    0.6358       157
            rose     0.4779    0.3624    0.4122       149
       sunflower     0.7125    0.7451    0.7284       153
           tulip     0.6322    0.7051    0.6667       156
      water_lily     0.4804    0.6622    0.5568       148

        accuracy                         0.5407      2062
       macro avg     0.5678    0.5437    0.5139      2062
    weighted

## FLOWER LEAKAGE

In [35]:
import random

SEED = 42
random.seed(SEED)

# =========================================================
# TẠO CONTROLLED DATA LEAKAGE
# =========================================================

leak_fraction = 0.50

# Số ảnh test sẽ bị leak vào train
num_leaked = int(
    len(test_idx) * leak_fraction
)

print("Clean Train size :", len(train_idx))
print("Validation size  :", len(val_idx))
print("Test size        :", len(test_idx))
print("Ảnh Test bị leak :", num_leaked)

Clean Train size : 9542
Validation size  : 2038
Test size        : 2062
Ảnh Test bị leak : 1031


## Chọn ngẫu nhiên 50% ảnh Test

In [36]:
# Copy để không thay đổi split CLEAN ban đầu
test_idx_copy = test_idx.copy()
train_idx_copy = train_idx.copy()

random.shuffle(test_idx_copy)
random.shuffle(train_idx_copy)

# Những ảnh test cố tình cho model nhìn thấy
leaked_test_ids = test_idx_copy[
    :num_leaked
]

# Giữ kích thước train không đổi:
# bỏ bớt num_leaked ảnh train bình thường
remaining_train_ids = train_idx_copy[
    :len(train_idx) - num_leaked
]

# Train LEAKAGE
leak_train_idx = (
    remaining_train_ids
    +
    leaked_test_ids
)

print(
    "Leak Train size:",
    len(leak_train_idx)
)

Leak Train size: 9542


## Ví dụ
### CLEAN
- Train: A B C D E F

- Test: X Y Z W

## LEAKAGE
- Train A B C X Y F

- Test X Y Z W

--> X, Y cùng xuất hiện ở cả Train Và Test

## Chứng minh thật sự có Leakage

In [37]:
overlap_train_test = (
    set(leak_train_idx)
    &
    set(test_idx)
)

print("=" * 55)
print("KIỂM TRA DATA LEAKAGE")
print("=" * 55)

print(
    "Số ảnh trùng Train/Test:",
    len(overlap_train_test)
)

print(
    "Tổng số ảnh Test:",
    len(test_idx)
)

print(
    "Leakage Ratio:",
    len(overlap_train_test)
    / len(test_idx)
)

print(
    "Leakage Ratio (%):",
    100
    * len(overlap_train_test)
    / len(test_idx)
)

KIỂM TRA DATA LEAKAGE
Số ảnh trùng Train/Test: 1031
Tổng số ảnh Test: 2062
Leakage Ratio: 0.5
Leakage Ratio (%): 50.0


## Tạo dataset LEAKAGE

In [38]:
leak_train_dataset = TransformSubset(
    base_dataset,
    leak_train_idx,
    train_transform
)

leak_val_dataset = TransformSubset(
    base_dataset,
    val_idx,
    eval_transform
)

leak_test_dataset = TransformSubset(
    base_dataset,
    test_idx,
    eval_transform
)

## DataLoader cho LEAKAGE

In [39]:
from torch.utils.data import DataLoader

leak_train_loader = DataLoader(
    leak_train_dataset,
    batch_size=32,
    shuffle=True
)

leak_val_loader = DataLoader(
    leak_val_dataset,
    batch_size=64,
    shuffle=False
)

leak_test_loader = DataLoader(
    leak_test_dataset,
    batch_size=64,
    shuffle=False
)

print(
    "Leak Train batches:",
    len(leak_train_loader)
)

print(
    "Leak Val batches:",
    len(leak_val_loader)
)

print(
    "Leak Test batches:",
    len(leak_test_loader)
)

Leak Train batches: 299
Leak Val batches: 32
Leak Test batches: 33


## Tạo & Train CNN

In [41]:
model_leak = model

In [42]:
model_leak = FlowerCNN(
    num_classes=num_classes
).to(device)

print(model_leak)

FlowerCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

## Loss và Optimizer giống CLEAN

In [43]:
criterion_leak = nn.CrossEntropyLoss()

optimizer_leak = torch.optim.AdamW(
    model_leak.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [44]:
import copy

num_epochs_leak = 5

best_val_acc_leak = 0.0
best_model_state_leak = None

print("=" * 70)
print("TRAIN CNN FLOWER - LEAKAGE")
print("=" * 70)

for epoch in range(num_epochs_leak):

    train_loss, train_acc = train_one_epoch(
        model_leak,
        leak_train_loader,
        optimizer_leak,
        criterion_leak
    )

    val_loss, val_acc = evaluate(
        model_leak,
        leak_val_loader,
        criterion_leak
    )

    print(
        f"Epoch {epoch+1:02d}/{num_epochs_leak} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc_leak:

        best_val_acc_leak = val_acc

        best_model_state_leak = copy.deepcopy(
            model_leak.state_dict()
        )


# Load model tốt nhất dựa trên VALIDATION
model_leak.load_state_dict(
    best_model_state_leak
)

print()
print("Training LEAKAGE hoàn tất!")

print(
    f"Best Validation Accuracy: "
    f"{best_val_acc_leak:.4f}"
)

TRAIN CNN FLOWER - LEAKAGE
Epoch 01/5 | Train Loss: 1.8886 | Train Acc: 0.3307 | Val Loss: 1.5586 | Val Acc: 0.4514
Epoch 02/5 | Train Loss: 1.6085 | Train Acc: 0.4293 | Val Loss: 1.3864 | Val Acc: 0.5098
Epoch 03/5 | Train Loss: 1.4377 | Train Acc: 0.4933 | Val Loss: 1.2379 | Val Acc: 0.5658
Epoch 04/5 | Train Loss: 1.3541 | Train Acc: 0.5246 | Val Loss: 1.1665 | Val Acc: 0.5829
Epoch 05/5 | Train Loss: 1.2754 | Train Acc: 0.5541 | Val Loss: 1.1443 | Val Acc: 0.5937

Training LEAKAGE hoàn tất!
Best Validation Accuracy: 0.5937


## Test Flower LEAKAGE

In [45]:
test_loss_leak, test_acc_leak = evaluate(
    model_leak,
    leak_test_loader,
    criterion_leak
)

print("=" * 55)
print("FLOWER LEAKAGE - FINAL TEST")
print("=" * 55)

print(
    f"Test Loss     : {test_loss_leak:.4f}"
)

print(
    f"Test Accuracy : {test_acc_leak:.4f}"
)

print(
    f"Test Accuracy : {test_acc_leak * 100:.2f}%"
)

FLOWER LEAKAGE - FINAL TEST
Test Loss     : 1.0949
Test Accuracy : 0.6232
Test Accuracy : 62.32%


## Tách Test thành SEEN và UNSEEN
- TEST
  + SEEN: Những ảnh đã bị đưa vào Train
  + UNSEEN: Những ảnh model chưa từng nhìn thấy

In [46]:
leaked_set = set(
    leaked_test_ids
)

seen_test_idx = [
    idx
    for idx in test_idx
    if idx in leaked_set
]

unseen_test_idx = [
    idx
    for idx in test_idx
    if idx not in leaked_set
]

print(
    "Seen Test:",
    len(seen_test_idx)
)

print(
    "Unseen Test:",
    len(unseen_test_idx)
)

Seen Test: 1031
Unseen Test: 1031


## Dataset Seen / Unseen

In [47]:
seen_test_dataset = TransformSubset(
    base_dataset,
    seen_test_idx,
    eval_transform
)

unseen_test_dataset = TransformSubset(
    base_dataset,
    unseen_test_idx,
    eval_transform
)

### Loader

In [48]:
seen_test_loader = DataLoader(
    seen_test_dataset,
    batch_size=64,
    shuffle=False
)

unseen_test_loader = DataLoader(
    unseen_test_dataset,
    batch_size=64,
    shuffle=False
)

## Đánh giá SEEN và UNSEEN

In [49]:
seen_loss, seen_acc = evaluate(
    model_leak,
    seen_test_loader,
    criterion_leak
)

unseen_loss, unseen_acc = evaluate(
    model_leak,
    unseen_test_loader,
    criterion_leak
)

### Kết quả

In [50]:
print("=" * 60)
print("PHÂN TÍCH DATA LEAKAGE")
print("=" * 60)

print(
    f"SEEN Test Accuracy   : "
    f"{seen_acc:.4f} "
    f"({seen_acc * 100:.2f}%)"
)

print(
    f"UNSEEN Test Accuracy : "
    f"{unseen_acc:.4f} "
    f"({unseen_acc * 100:.2f}%)"
)

print(
    f"Difference           : "
    f"{(seen_acc - unseen_acc) * 100:.2f}%"
)

PHÂN TÍCH DATA LEAKAGE
SEEN Test Accuracy   : 0.6421 (64.21%)
UNSEEN Test Accuracy : 0.6043 (60.43%)
Difference           : 3.78%


## So sánh CLEAN và LEAKAGE

In [51]:
print("=" * 65)
print("FLOWER: CLEAN vs DATA LEAKAGE")
print("=" * 65)

print(
    f"CLEAN Test Accuracy   : "
    f"{test_acc_clean * 100:.2f}%"
)

print(
    f"LEAKAGE Test Accuracy : "
    f"{test_acc_leak * 100:.2f}%"
)

print()

print(
    f"LEAKED-SEEN Accuracy  : "
    f"{seen_acc * 100:.2f}%"
)

print(
    f"LEAKED-UNSEEN Accuracy: "
    f"{unseen_acc * 100:.2f}%"
)

FLOWER: CLEAN vs DATA LEAKAGE
CLEAN Test Accuracy   : 54.07%
LEAKAGE Test Accuracy : 62.32%

LEAKED-SEEN Accuracy  : 64.21%
LEAKED-UNSEEN Accuracy: 60.43%


In [52]:
import pandas as pd

results = pd.DataFrame({
    "Experiment": [
        "Flower CLEAN",
        "Flower LEAKAGE - Full Test",
        "Flower LEAKAGE - Seen Test",
        "Flower LEAKAGE - Unseen Test"
    ],

    "Accuracy": [
        test_acc_clean,
        test_acc_leak,
        seen_acc,
        unseen_acc
    ]
})

results["Accuracy (%)"] = (
    results["Accuracy"] * 100
)

results

,Experiment,Accuracy,Accuracy (%)
0,Flower CLEAN,0.540737,54.073715
1,Flower LEAKAGE - Full Test,0.623181,62.318138
2,Flower LEAKAGE - Seen Test,0.642095,64.209505
3,Flower LEAKAGE - Unseen Test,0.604268,60.426770


| Thí nghiệm                       |   Accuracy | Nhận xét ngắn                                                                          |
| -------------------------------- | ---------: | -------------------------------------------------------------------------------------- |
| **Flower CLEAN**                 | **54.07%** | Kết quả đáng tin cậy vì Test không xuất hiện trong Train.                              |
| **Flower LEAKAGE – Full Test**   | **62.32%** | Tăng khoảng **8.24 điểm %** so với CLEAN do một phần Test đã bị đưa vào Train.         |
| **Flower LEAKAGE – Seen Test**   | **64.21%** | Cao nhất vì đây là các ảnh Test mà model đã từng nhìn thấy khi training.               |
| **Flower LEAKAGE – Unseen Test** | **60.43%** | Thấp hơn Seen khoảng **3.78 điểm %**, cho thấy model có lợi thế trên dữ liệu bị rò rỉ. |


### **Kết luận:** Data Leakage làm kết quả đánh giá trở nên lạc quan hơn thực tế. Accuracy cao hơn không có nghĩa model generalize tốt hơn, vì Test đã không còn độc lập với quá trình training.